# Publish to the SDL from a manifest

Publishes a Tsunami-HySEA experiment to the CINECA **Simulation Data Lake** from an existing
`sdl_manifest.yml` (create/fill one with **Generate manifest & publish**, or by hand).

**Setup → Login → choose a manifest → (optionally) Preview → Publish.** You pick **one** manifest at a
time; nothing is uploaded until you press *Publish*. A **per-file** upload bar (bytes) plus an
**overall** bar (files N/total) are shown. The login server is used (a manifest's own `sdl.server` is overridden).

In [ ]:
import os, sys, glob, yaml
for p in ("~/HySEALab/Notebooks/01_sdl_publish", "~/SDL", "~/HySEALab/Notebooks", "."):
    pp = os.path.expanduser(p)
    if os.path.exists(os.path.join(pp, "sdl_lib.py")):
        sys.path.insert(0, pp); break
import importlib, sdl_lib; importlib.reload(sdl_lib)
import ipywidgets as W
from IPython.display import display
STATE = {}
print("sdl_lib loaded from:", os.path.dirname(sdl_lib.__file__))

In [ ]:
srv = W.Dropdown(options=[("prod", "prod"), ("dev", "dev")], value="prod", description="Server:")
u   = W.Text(description="SDL user:", layout=W.Layout(width="380px"))
pw  = W.Password(description="Password:", layout=W.Layout(width="380px"))
b   = W.Button(description="Login", button_style="primary")
o   = W.Output()

def _login(_):
    with o:
        o.clear_output()
        base = sdl_lib.SERVERS.get(srv.value, sdl_lib.SERVERS["prod"])
        try:
            tok = sdl_lib.get_auth_token(base, u.value, pw.value)
            STATE["token"]   = tok
            STATE["headers"] = sdl_lib.auth_headers(tok)
            STATE["base"]    = base
            print("Login OK on", base)
        except Exception as e:
            print("Login failed:", type(e).__name__, e)

b.on_click(_login)
display(W.VBox([srv, u, pw, b, o]))

In [ ]:
base   = W.Text(value="~", description="Search in:", layout=W.Layout(width="470px"))
find   = W.Button(description="🔍 Find", layout=W.Layout(width="120px"))
sel    = W.Select(options=[], description="Manifest:", rows=6, layout=W.Layout(width="780px"))
manual = W.Text(value="", description="or path:",
                placeholder="full path to an sdl_manifest.yml (overrides the selection above)",
                layout=W.Layout(width="780px"))
findout = W.Output()

def _chosen():
    return os.path.expanduser(manual.value.strip()) if manual.value.strip() else (sel.value or "")

def _find(_):
    with findout:
        findout.clear_output()
        root = os.path.expanduser(base.value.strip() or "~")
        hits = sorted(glob.glob(os.path.join(root, "**", "sdl_manifest.yml"), recursive=True))
        sel.options = hits
        print("found %d manifest(s) under %s" % (len(hits), root))
find.on_click(_find)

prev    = W.Button(description="👁 Preview manifest", layout=W.Layout(width="200px"))
pub     = W.Button(description="Publish selected manifest", button_style="primary",
                   layout=W.Layout(width="280px"))
prevout = W.Output()
out     = W.Output()

def _preview(_):
    with prevout:
        prevout.clear_output()
        mf = _chosen()
        if not mf or not os.path.exists(mf):
            print("Pick a manifest in the list (or type a valid path)."); return
        print("# " + mf + "\n")
        print(open(mf).read())
prev.on_click(_preview)

def _expected_uploads(mf):
    m = yaml.safe_load(open(mf))
    root = os.path.abspath(os.path.join(os.path.dirname(mf), (m.get("project") or {}).get("root", ".")))
    g = [f for f in os.listdir(os.path.join(root, "bathy"))    if f.endswith((".grd", ".nc"))]
    p = [f for f in os.listdir(os.path.join(root, "parfiles")) if f.lower().endswith((".txt", ".par"))]
    s = os.listdir(os.path.join(root, "sim"))
    return len(g) + 2 + len(p) + len(s) + 1     # grids + 2 plots + parfiles + sims + README

def _publish(_):
    with out:
        out.clear_output()
        if not STATE.get("token"):
            print("Login first (run the Login cell)."); return
        mf = _chosen()
        if not mf or not os.path.exists(mf):
            print("Pick a manifest in the list (or type a valid path)."); return
        name = os.path.basename(os.path.dirname(mf)) or os.path.basename(mf)
        try:
            tot = _expected_uploads(mf)
        except Exception:
            tot = 0
        bar = W.IntProgress(value=0, min=0, max=max(tot, 1), description=name,
                            bar_style="info", layout=W.Layout(width="560px"))
        lab = W.Label(value=("0/%d files" % tot) if tot else "uploading...")
        display(W.VBox([bar, lab]))
        def _log(msg, _b=bar, _l=lab, _t=tot):
            print(msg)
            parts = str(msg).strip().split(None, 1)
            if len(parts) == 2 and parts[0] == "✅" and " " not in parts[1]:
                if _t and _b.value < _t: _b.value += 1
                _l.value = ("%d/%d files — %s" % (_b.value, _t, parts[1][:60])) if _t else parts[1][:70]
        try:
            r = sdl_lib.publish_from_manifest(mf, token=STATE["token"], headers=STATE["headers"],
                                              base_url=STATE["base"], log=_log)
            bar.value = bar.max; bar.bar_style = "success"
            lab.value = "DONE — experiment %s  version %s" % (r.get("experiment_id"), r.get("version"))
            print("✅ %s -> experiment %s version %s : %s/app/"
                  % (name, r.get("experiment_id"), r.get("version"), r.get("base_url")))
        except Exception as e:
            bar.bar_style = "danger"
            print("❌ %s: %s: %s" % (name, type(e).__name__, e))
pub.on_click(_publish)

display(W.VBox([W.HBox([base, find]), findout, sel, manual, W.HBox([prev, pub]), prevout, out]))